# A Simple Bank System
----------------------
### Objective: 
    To create a Python program that manages bank accounts using Classes and Objects.
    
## Part A: The Goal
    You need to write a program that acts like a digital bank. Users should be able to create an account, save their money, and check their balance using a simple menu.

## Part B: Program Requirements
### 1. The "Bank" Class
    Create a class called Bank. This class will be the "blueprint" for every account. It must have:
    • Attributes:
        o Name: To store the account holder's name.
        o Balance: To store how much money is in the account.
        o Account Number: A unique 16-digit number generated automatically using the random module.
        
    • Actions (Methods):
        o Deposit: Add a specific amount of money to the balance.
        o Withdraw: Subtract money from the balance (only if the user has enough).
        o Show Details: Print the name, balance, and account number to the screen.

### 2. Main Menu Logic
    Your program should run in a loop and ask the user to choose an option:
        1. Create Account: Ask for a name and starting money. Create the account and save it 
        in a list.
        2. Deposit Money: Ask for an account number. If it exists, add money to it.
        3. Withdraw Money: Ask for an account number. If it exists, let the user take money 
        out.
        4. View Details: Ask for an account number and show all the info for that account.
        5. Exit: Close the program with a "Thank You" message.

## Part C: Rules & Constraints
    • Storage: All created accounts must be stored in a list so the program can find them later.
    • Search: Before doing a deposit or withdrawal, the program must search the list to make sure the account number is correct.
    • Safety: If an account number is not found, display the message: "Account number does not match".

In [1]:
# Custom Exceptions Classes
class InvalidAmountError(Exception):
    """Raised when an amount is zero, negative, or invalid."""
    pass

class InsufficientBalanceError(Exception):
    """Raised when withdrawal amount is greater than balance."""
    pass

class AccountNotFoundError(Exception):
    """Raised when an account number does not exist."""
    pass

class DuplicateAccountError(Exception):
    """Raised when a generated account number already exists."""
    pass

class InvalidNameError(Exception):
    """Raised when the account holder name is invalid."""
    pass
    
class InvalidAccountNumberError(Exception):
    """Raised when the account number format is invalid."""
    pass

In [2]:
# Custom Validation Functions

# Validate Name
def validate_name(name): 
    """
    Validate account holder name.
    
    Rules:
    - Must not be empty.
    - Must contain only alphabets and spaces.
    """ 
    
    name = name.strip()
    
    if not name:
        raise InvalidNameError( "Name cannot be empty." )
        
    if not all(char.isalpha() or char.isspace() for char in name):
        raise InvalidNameError( "Name must contain only letters and spaces." )
        
    return name


# Validate Account Number
def validate_account_number(account_number):
    """
    Validate account number.
    
    Rules:
    - Must be an integer.
    - Must contain exactly 16 digits.
    """
    
    if not isinstance(account_number, int):
        raise InvalidAccountNumberError( "Account number must be an integer." )
        
    if len(str(account_number)) != 16:
        raise InvalidAccountNumberError( "Account number must contain exactly 16 digits." )
    
    return account_number


# Validate Money Amount 
def validate_amount(amount):
    """
    Validate money amount.
    
    Rules:
    - Must be an integer.
    - Must be greater than zero.
    """
    
    if not isinstance(amount, int):
        raise InvalidAmountError( "Amount must be an integer." )
        
    if amount <= 0:
        raise InvalidAmountError( "Amount must be greater than zero." )
        
    return amount

In [3]:
# Abstraction

import random
from abc import ABC, abstractmethod

class BankAccount(ABC):

    def __init__(self, name, balance):
        # Encapsulation: private attributes
        self.__name = validate_name(name) # Validate 'name' before storing data
        self.__balance = validate_amount(balance)

        # Generate a unique 16-digit account number
        self.__account_number = self.__generate_account_number()

    def __generate_account_number(self):
        """Generate a 16-digit account number."""
        return random.randint(10**15, 10**16 - 1)

    
    # Getter methods
    def get_name(self):
        return self.__name

    def get_balance(self):
        return self.__balance

    def get_account_number(self):
        return self.__account_number

    
    # Protected method: used internally to update balance
    def _set_balance(self, balance):
        self.__balance = balance

    
    # Abstract methods
    @abstractmethod
    def deposit(self, amount):
        pass

    @abstractmethod
    def withdraw(self, amount):
        pass

    @abstractmethod
    def show_details(self):
        pass

In [4]:
# Inheritance + Polymorphism

class Bank(BankAccount): # Inherit Abstract Class

    def __init__(self, name, balance):
        # Constructor of parent class
        super().__init__(name, balance)

    
    def deposit(self, amount):
        # Validate deposit amount
        amount = validate_amount(amount)
            
        new_balance = self.get_balance() + amount
        self._set_balance(new_balance)
        
        print("Money deposited successfully.")
        print(f"New Balance: Rs.{self.get_balance():.2f}")

    
    def withdraw(self, amount):
         # Validate deposit amount
        amount = validate_amount(amount)
        
        if amount > self.get_balance():
            raise InsufficientBalanceError("Withdraw amount must not exceed available balance.")
        
        new_balance = self.get_balance() - amount
        self._set_balance(new_balance)
        
        print("Money withdrawn successfully.")
        print(f"New Balance: Rs.{self.get_balance():.2f}")

    
    def show_details(self):
        print("\n----- Account Details -----")
        print(f"Name           : {self.get_name()}")
        print(f"Account Number : {self.get_account_number()}")
        print(f"Balance        : Rs.{self.get_balance():.2f}")
        print("-----------------------------")

In [5]:
# Generate Unique Account Number

# Account Storage
accounts = []

def create_unique_account(name, balance):
    while True:
        account = Bank(
            name,
            balance
        )

        # Check whether generated account number already exists
        duplicate = False

        for existing_account in accounts:
            if (existing_account.get_account_number() == account.get_account_number()):
                duplicate = True
                break

        if not duplicate:
            accounts.append(account)
            return account

        # Very unlikely, but handled safely of new account number
        raise DuplicateAccountError("Unable to generate a unique account number.")

In [6]:
# Find Account

def find_account(account_number):
    # Validate account number first
    account_number = validate_account_number(account_number)

    # Search account list
    # loop to check account number exists or not
    for account in accounts:
        if account.get_account_number() == account_number:
            return account

    # if account not found
    raise AccountNotFoundError("Account number does not match")

In [7]:
# Create New Account

def create_account():
    try:
        name = input("Enter account holder name: ")
        # Custom name validation
        name = validate_name(name)

        # Read starting money as STRING first
        starting_money_input = input("Enter starting money: ").strip()

        # Make sure input contains only digits
        if not starting_money_input.isdigit():
            raise InvalidAmountError("Starting money must contain only integer digits.")

        # Convert string to integer
        starting_money = int(starting_money_input)

        # Validate amount
        starting_money = validate_amount(starting_money)

        account = create_unique_account(
            name,
            starting_money
        )

        print("\nAccount created successfully!")
        print(f"Name           : {account.get_name()}")
        print(f"Account Number : {account.get_account_number()}")
        print(f"Balance        : Rs.{account.get_balance():.2f}")

    except InvalidNameError as e:
        print(f"Error: {e}")

    except InvalidAmountError as e:
        print(f"Error: {e}")

    except DuplicateAccountError as e:
        print(f"Error: {e}")

    except Exception as e:
        print(f"Unexpected error: {e}")

In [8]:
# Get and Validate Account Number

def get_valid_account_number():
    account_number_input = input("Enter account number: ").strip()
    
    # Check whether input contains only digits 
    if not account_number_input.isdigit():
        raise InvalidAccountNumberError("Account number must contain only digits.")
        
    # Convert string to integer
    account_number = int(account_number_input)

    # Validate 16-digit account number
    # Validate and RETURN the account number
    return validate_account_number(account_number)

In [9]:
#Get and Validate Money Amount
def get_valid_amount(message):
    amount_input = input( message ).strip()
    # Check whether input contains digits only
    if not amount_input.isdigit():
        raise InvalidAmountError( "Amount must contain only integer digits." )
        
    # Convert string to integer
    amount = int(amount_input) 
    
    # Validate amount
    return validate_amount(amount)

In [10]:
# Deposit Money

def deposit_money():
    try:
        account_number = get_valid_account_number()

        account = find_account(account_number)

        amount = get_valid_amount("Enter amount to deposit: ")

        account.deposit(amount)

    except AccountNotFoundError as e:
        print(f"Error: {e}")

    except InvalidAccountNumberError as e:
        print(f"Error: {e}")

    except InvalidAmountError as e:
        print(f"Error: {e}")

    except Exception as e:
        print(f"Unexpected error: {e}")

In [11]:
# Withdraw Money

def withdraw_money():
    try:
        account_number = get_valid_account_number()

        account = find_account(account_number)

        amount = get_valid_amount("Enter amount to withdraw: ")

        account.withdraw(amount)

    except AccountNotFoundError as e:
        print(f"Error: {e}")

    except InvalidAccountNumberError as e:
        print(f"Error: {e}")

    except InvalidAmountError as e:
        print(f"Error: {e}")

    except InsufficientBalanceError as e:
        print(f"Error: {e}")

    except Exception as e:
        print(f"Unexpected error: {e}")

In [12]:
# View Account Details

def view_details():
    try:
        account_number = get_valid_account_number()

        account = find_account(account_number)

        account.show_details()

    except AccountNotFoundError as e:
        print(f"Error: {e}")

    except InvalidAccountNumberError as e:
        print(f"Error: {e}")

    except Exception as e:
        print(f"Unexpected error: {e}")

In [13]:
# Main Menu

while True:
    print("\n========== BANK SYSTEM ==========")
    print("1. Create Account")
    print("2. Deposit Money")
    print("3. Withdraw Money")
    print("4. View Details")
    print("5. Exit")
    print("=================================")

    choice = input("Enter your choice: ").strip()

    try:
        if choice == "1":
            create_account()

        elif choice == "2":
            deposit_money()

        elif choice == "3":
            withdraw_money()

        elif choice == "4":
            view_details()

        elif choice == "5":
            print("\nThank You for using the Bank System!")
            break

        else:
            print("Invalid choice. Please select 1-5.")

    except KeyboardInterrupt:
        print("\n\nProgram interrupted.")
        print("Thank You for using the Bank System!")
        break

    except Exception as e:
        print(f"Unexpected error: {e}")


========== BANK SYSTEM ==========
1. Create Account
2. Deposit Money
3. Withdraw Money
4. View Details
5. Exit


Enter your choice:  4
Enter account number:  asdf123


Error: Account number must contain only digits.

========== BANK SYSTEM ==========
1. Create Account
2. Deposit Money
3. Withdraw Money
4. View Details
5. Exit


Enter your choice:  4
Enter account number:  122345


Error: Account number must contain exactly 16 digits.

========== BANK SYSTEM ==========
1. Create Account
2. Deposit Money
3. Withdraw Money
4. View Details
5. Exit


...

Money withdrawn successfully.
New Balance: Rs.900.00

========== BANK SYSTEM ==========
1. Create Account
2. Deposit Money
3. Withdraw Money
4. View Details
5. Exit


Enter your choice:  4
Enter account number:  5383597282898952



----- Account Details -----
Name           : Ritesh
Account Number : 5383597282898952
Balance        : Rs.900.00
-----------------------------

========== BANK SYSTEM ==========
1. Create Account
2. Deposit Money
3. Withdraw Money
4. View Details
5. Exit


Enter your choice:  5



Thank You for using the Bank System!
